In [ ]:
#imports
import numpy as np
import cv2
import time
from matplotlib import pyplot as plt
from IPython.display import display,clear_output

In [ ]:
def createRegionMask(image):
    """Create mask for region of interest - the area in front of the vehicle"""
    height, width = image.shape[:2]
    mask = np.zeros((height, width), dtype=np.uint8)
    
    # Define points for a wider and expanded trapezoid
    regionMaskVertices = np.array([
        [0, height], # Bottom left 
        [width * 0.35, height * 0.5], # Top left 
        [width * 0.65, height * 0.5], # Top right
        [width, height] # Bottom right
    ], np.int32)
    
    cv2.fillPoly(mask, [regionMaskVertices], 255)
    return mask

In [ ]:
def enhanceLaneLines(image):
    """Enhance lane line detection - focus on yellow and white lines"""
    # Convert to HSV for better color detection
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    
    # Define ranges for yellow lines
    yellowLowerRange = np.array([15, 40, 40])
    yellowUpperRange = np.array([35, 255, 255])
    yellowMask = cv2.inRange(hsv, yellowLowerRange, yellowUpperRange)
    
    # Define ranges for white lines
    whiteLower = np.array([0, 0, 200])
    whiteUpper = np.array([255, 30, 255])
    whiteMask = cv2.inRange(hsv, whiteLower, whiteUpper)
    
    # Combine masks
    combinedMask = cv2.bitwise_or(yellowMask, whiteMask)
    
    # Apply mask to original image
    maskedImage = cv2.bitwise_and(image, image, mask=combinedMask)
    
    return maskedImage, combinedMask

In [ ]:
def filterLinesByPosition(lines, imageWidth, imageCenter):
    """Filter lines based on their position relative to the center of the vehicle"""
    if lines is None:
        return [], []
    
    leftLines = []
    rightLines = []
    
    for line in lines:
        x1, y1, x2, y2 = line[0]
        
        # Calculate slope
        if x2 - x1 == 0:
            continue
        slope = (y2 - y1) / (x2 - x1)
        
        # Filter lines that are too horizontal or too steep
        if abs(slope) < 0.2 or abs(slope) > 3:
            continue
        
        # Calculate midpoint of the line
        midX = (x1 + x2) / 2
        
        # Calculate where the line intersects the bottom of the image
        intercept = y1 - slope * x1
        bottomX = (600 - intercept) / slope if slope != 0 else midX
        
        # Improved classification for left and right lines
        if slope < -0.2:  # Negative slope = left line
            if bottomX < imageCenter + 100:  # Some tolerance
                leftLines.append((slope, intercept))
        elif slope > 0.2:  # Positive slope = right line
            if bottomX > imageCenter - 100:  # Some tolerance
                rightLines.append((slope, intercept))
    
    return leftLines, rightLines

In [ ]:
def makeCoordinates(image, slope, intercept):
    """Create coordinates for a line based on slope and intercept - extended to full height"""
    y1 = image.shape[0]  # Bottom of image
    y2 = int(y1 * 0.5)   # Middle of image (instead of 0.6)
    
    if abs(slope) < 0.001:
        slope = 0.001 if slope >= 0 else -0.001
        
    x1 = int((y1 - intercept) / slope)
    x2 = int((y2 - intercept) / slope)
    
    # Ensure coordinates are within image bounds
    x1 = max(0, min(x1, image.shape[1] - 1))
    x2 = max(0, min(x2, image.shape[1] - 1))
    
    return np.array([x1, y1, x2, y2])


In [ ]:
def averageSlopeIntercept(image, lines):
    """Calculate average of slopes and intercepts for lane lines"""
    imageCenter = image.shape[1] // 2
    leftLines, rightLines = filterLinesByPosition(lines, image.shape[1], imageCenter)
    
    outputLines = []
    
    # Process left line
    if len(leftLines) > 0:
        avgSlope, avgIntercept = np.mean(leftLines, axis=0)
        coords = makeCoordinates(image, avgSlope, avgIntercept)
        outputLines.append(coords)
    
    # Process right line
    if len(rightLines) > 0:
        avgSlope, avgIntercept = np.mean(rightLines, axis=0)
        coords = makeCoordinates(image, avgSlope, avgIntercept)
        outputLines.append(coords)
    
    return outputLines

In [ ]:
def drawLines(image, lines):
    """Draw lane lines on the image"""
    lineImg = np.zeros_like(image)
    
    if lines is not None and len(lines) > 0:
        for line in lines:
            x1, y1, x2, y2 = line
            cv2.line(lineImg, (x1, y1), (x2, y2), (0, 255, 0), 8)
    
    return cv2.addWeighted(image, 0.8, lineImg, 1, 1)

In [ ]:
def drawLaneArea(image, lines):
    """Draw current lane area - improved"""
    if len(lines) == 2:  # There is a left and right line
        #laneImg = np.zeros_like(image)
        laneImg = np.zeros_like(image)
        
        leftLine = lines[0]
        rightLine = lines[1]
        
        # Ensure the left line is actually on the left and right on the right
        if leftLine[0] > rightLine[0]:  # If the "left" line is more to the right
            leftLine, rightLine = rightLine, leftLine
        
        # Create polygon from lane lines
        pts = np.array([
            [leftLine[0], leftLine[1]],   # Bottom left
            [leftLine[2], leftLine[3]],   # Top left
            [rightLine[2], rightLine[3]], # Top right
            [rightLine[0], rightLine[1]]  # Bottom right
        ], np.int32)
        
        cv2.fillPoly(laneImg, [pts], (0, 255, 255))
        return cv2.addWeighted(image, 0.7, laneImg, 0.3, 0)
    
    return image


In [ ]:
def calculateCurvature(edges, image_center):
    y_vals, x_vals = np.nonzero(edges)
    if len(y_vals) < 50:
        return "Curvature: Straight"
        
    # check the right or left side separately so that the calculation does not get confused
    left_mask = x_vals < image_center
    right_mask = x_vals >= image_center
    
    for mask in [left_mask, right_mask]:
        y_m = y_vals[mask]
        x_m = x_vals[mask]
        if len(y_m) > 30:
            # Fit a polynomial of degree 2
            fit = np.polyfit(y_m, x_m, 2)
            A = fit[0]
            
            if abs(A) < 1e-4: # A low value means the edge is straight.
                continue
                
            # Curve radius formula. Calculated at the bottom of the image (y=600)
            radius_pixels = ((1 + (2 * A * 600 + fit[1])**2)**1.5) / np.absolute(2 * A)
            
            # Approximate conversion from pixels to meters (according to standard proportions)
            radius_meters = int(radius_pixels * (30 / 600)) 
            
            if radius_meters > 1500:
                return "Curvature: Straight"
                
            direction = "Right" if A > 0 else "Left"
            return f"Curvature: {direction} Curve (Radius: {radius_meters}m)"
            
    return "Curvature: Straight"

In [ ]:
def processImage(img):
    """Process image for lane detection (Adapted for video frames)"""

    #Resize the image to a fixed resolution (800x600) for consistent processing speed and results
    img = cv2.resize(img, (800, 600))
    
    # Enhance specific colors (usually white and yellow) to make lane lines stand out
    enhancedImg, colorMask = enhanceLaneLines(img)
    
    # Generate a mask to focus only on the region of interest (typically the lower half of the road)
    regionInterestMask = createRegionMask(img)
    
    # Convert the color-enhanced image to grayscale for easier intensity processing
    gray = cv2.cvtColor(enhancedImg, cv2.COLOR_BGR2GRAY)
    
    # Apply CLAHE (Contrast Limited Adaptive Histogram Equalization) to improve local contrast, 
    # which helps in varying lighting conditions (e.g., shadows)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(gray)
    
    # Apply the region of interest mask to the enhanced grayscale image to ignore background noise
    roiEnhanced = cv2.bitwise_and(enhanced, regionInterestMask)
    
    # Perform Canny edge detection to find the outlines of the lane lines
    edges = cv2.Canny(roiEnhanced, 30, 100)
    
    # Use Probabilistic Hough Transform to detect straight line segments from the edges
    lines = cv2.HoughLinesP(
        edges, 
        rho=1,             # Distance resolution of the accumulator in pixels
        theta=np.pi/180,   # Angle resolution of the accumulator in radians
        threshold=15,      # Minimum number of intersections to detect a line
        minLineLength=25,  # Minimum length of a line segment in pixels
        maxLineGap=300     # Maximum allowed gap between segments to treat them as a single line
    )
    
    # Group the detected line segments into two distinct lines (left and right) by averaging their slopes and intercepts
    averageLines = averageSlopeIntercept(img, lines)
    
    # Draw a colored polygon filling the area between the two detected lane lines
    laneImage = drawLaneArea(img, averageLines)
    
    # Overlay the solid left and right lane lines onto the final image
    laneImage = drawLines(laneImage, averageLines)
    
    # Calculating the curve and printing on the image
    curve_text = calculateCurvature(edges, img.shape[1] // 2)
    cv2.putText(laneImage, curve_text, (30, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2, cv2.LINE_AA)

    # Return the final composite image ready for display
    return laneImage

In [ ]:
# reading the video and saving the result video file
print("starting to load the clara video ...")

video_filename = 'carla_video.mp4' 
cap = cv2.VideoCapture(video_filename)

# check if the video openes at all
if not cap.isOpened():
    print(f"Error: Could not find or open the file .'{video_filename}'!")
    print("Please make sure the file is in the same folder as the code and that its name is accurate..")
else:
    # Settings for saving the new video (using XVID and AVI this is faster)
    fourcc = cv2.VideoWriter_fourcc(*'XVID')
    out = cv2.VideoWriter('Final_Project_Submission.avi', fourcc, 30.0, (800, 600))
    
    frame_count = 0
    success_count = 0
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break # The video is over
            
        # Frame processing and path identification
        processed_frame = processImage(frame)
        
        # make sure the size matches exactly 800x600 before saving
        if processed_frame.shape[1] == 800 and processed_frame.shape[0] == 600:
            out.write(processed_frame)
            success_count += 1
        else:
            print(f"Warning: Invalid frame size: {processed_frame.shape}")
        
        frame_count += 1
        if frame_count % 30 == 0:
            print(f"processed {frame_count} Frames ...")
    

In [ ]:
    # Closing the files
    cap.release()
    out.release()
    
    if success_count > 0:
        print(f"Frames saved {success_count} Processing completed successfully .")
        print("the video file Final_Project_Submission.avi is ready.")
    else:
        print("Error: No frames saved to the new file.")